# BioMistral-7B + Knowledge Graph — Medical QA Evaluation

**Runtime required:** GPU (T4 or better)
Go to **Runtime → Change runtime type → GPU** before running.

## What this notebook does
1. Installs all dependencies
2. Uploads your project files + `medical_kg_v3.json`
3. Downloads **BioMistral/BioMistral-7B** locally — this is the *unlimited calls* path (no API rate limits)
4. Runs evaluation on the FirstAidQA dataset across three systems:
   - **Standalone BioMistral-7B** — the published medical LLM baseline
   - **Your Method: KG + BioMistral** — KG retrieval + BioMistral generation
   - **Pure KG** — KG retrieval only, no LLM
5. Optionally adds the three-way ablation (linguistic vs knowledge contribution)
6. Saves a publication-style results table and per-sample CSV

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅  GPU: {gpu} — {vram:.1f} GB VRAM')
    print('    4-bit quantisation will be used (≈4.5 GB) — fits comfortably.')
else:
    print('⚠️  No GPU detected!')
    print('   Go to Runtime → Change runtime type → GPU, then re-run.')
    print('   The pipeline will still run on CPU but will be very slow.')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# This takes ~2 minutes on a fresh Colab instance.
%pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece \
    rouge-score \
    bert-score \
    datasets \
    networkx \
    python-dotenv \
    huggingface_hub

print('✅  Dependencies installed.')

In [ ]:
# ── Cell 3: Upload project files ─────────────────────────────────────────────
# Upload the project zip (medical_kg_firstaid.zip) using the file chooser.
# It will be unzipped into the working directory.

from google.colab import files
import zipfile, os

print('Select your project zip file (medical_kg_firstaid.zip):')
uploaded = files.upload()

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('.')
        print(f'✅  Unzipped {fname}')

print('\nFiles in working directory:')
print([f for f in os.listdir('.') if f.endswith('.py') or f.endswith('.json')])

In [ ]:
# ── Cell 4: Upload medical_kg_v3.json ────────────────────────────────────────
# If your KG JSON is NOT inside the zip, upload it separately here.
# If it was already in the zip, skip this cell.

import os
if os.path.exists('medical_kg_v3.json'):
    import json
    with open('medical_kg_v3.json') as f:
        kg = json.load(f)
    stats = kg.get('statistics', {})
    print(f'✅  medical_kg_v3.json already present: '
          f"{stats.get('total_nodes','?')} nodes, {stats.get('total_edges','?')} edges")
else:
    print('medical_kg_v3.json not found — uploading now...')
    from google.colab import files
    uploaded = files.upload()
    print('✅  Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Cell 5: (Optional) Set API keys ──────────────────────────────────────────
# The pipeline prioritises the LOCAL BioMistral model (no API needed).
# API keys are only used:
#   - If the local model fails to load (no GPU / OOM)
#   - For the LLM-judge metrics (--llm_judge flag) if you want GPT-4o-mini scoring
# You can leave these blank and rely entirely on the local model.

import os

# Groq key — free at https://console.groq.com (used as API fallback for BioMistral
# and for Llama-3 baseline if you add it to --models)
os.environ['GROQ_API_KEY'] = ''   # ← paste your key here, or leave blank

# OpenAI key — only needed if you run --llm_judge with GPT-4o-mini scoring
os.environ['OPENAI_API_KEY'] = ''  # ← optional

print('API keys configured (blank = local-only mode).')

In [ ]:
# ── Cell 6: Download & load BioMistral-7B ────────────────────────────────────
# This downloads the model weights on first run (~14 GB transfer) and loads them
# in 4-bit quantisation (~4.5 GB VRAM).  Subsequent runs use the Colab cache.
# Once loaded, ALL three pipelines share this single copy — no duplicate loads.

import sys
sys.path.insert(0, '.')   # ensure project files are importable

from biomistral_backend import download_biomistral

bm = download_biomistral(load_in_4bit=True)
print()
print('Backend info:', bm)

In [ ]:
# ── Cell 7: Quick sanity test ────────────────────────────────────────────────
# Runs one question through all three pipelines to confirm everything works
# before the full evaluation.

import sys
sys.path.insert(0, '.')

TEST_Q = 'What should you do if someone is choking?'
KG_PATH = 'medical_kg_v3.json'

# 1. Standalone BioMistral
from biomistral_backend import get_biomistral_backend
bm = get_biomistral_backend()
ans_bm = bm.chat(
    system='You are a biomedical first aid expert. Answer concisely.',
    user=TEST_Q, max_new_tokens=120)
print('=== Standalone BioMistral ===')
print(ans_bm)

# 2. KG + BioMistral
from kg_pipeline3 import KGPipeline
kg_pipe = KGPipeline(kg_path=KG_PATH, prefer_local=True)
ans_kg = kg_pipe.generate_answer_only(TEST_Q)
print('\n=== KG + BioMistral ===')
print(ans_kg)

# 3. Pure KG
from kg_pipeline3 import PureKGPipeline
pure = PureKGPipeline(kg_path=KG_PATH)
ans_pure = pure.generate_answer_only(TEST_Q)
print('\n=== Pure KG ===')
print(ans_pure)

In [ ]:
# ── Cell 8: Full evaluation ───────────────────────────────────────────────────
# Runs all three systems on the FirstAidQA test split.
#
# Flags:
#   --n_samples 50   → evaluate on 50 samples (fast, good for a pilot run)
#                      remove the flag to run the full dataset
#   --no_bertscore   → skip BERTScore (saves ~10 min + 2 GB RAM on first run)
#                      remove it once you want the full metric set
#   --backend local  → use local BioMistral (unlimited calls, recommended)

!python evaluate_pipeline3.py \
    --backend local \
    --models biomistral kg kg_pure \
    --kg_path medical_kg_v3.json \
    --n_samples 50 \
    --no_bertscore \
    --output_dir results

In [ ]:
# ── Cell 9: Add BERTScore (run after Cell 8 if you skipped it) ───────────────
# BERTScore-F1 is the most informative metric for open-ended medical QA.
# Only run this once Cell 8 is complete.

!python evaluate_pipeline3.py \
    --backend local \
    --models biomistral kg kg_pure \
    --kg_path medical_kg_v3.json \
    --n_samples 50 \
    --output_dir results_with_bertscore

In [ ]:
# ── Cell 10: Three-way ablation (KG Knowledge vs Linguistic contribution) ─────
# Runs Pure KG / KG+BioMistral(constrained) / KG+BioMistral(full) side by side
# and decomposes the score improvement into linguistic and knowledge components.

import sys
sys.path.insert(0, '.')

from kg_constrained import run_standalone_eval
results = run_standalone_eval(
    kg_path='medical_kg_v3.json',
    n_samples=50,
    groq_key=''   # leave blank to rely on local model
)

In [ ]:
# ── Cell 11: LLM-as-judge metrics (optional — needs OpenAI or Groq key) ──────
# Adds 7 clinical quality dimensions: Escalation-Accuracy, Red-Flag-Recall,
# Unsafe-Rate, Hallucination-Rate, Guideline-Adherence, Explainability,
# Logical-Consistency.
# Uses GPT-4o-mini if OPENAI_API_KEY is set, else falls back to Groq-70B.

OPENAI_KEY = os.environ.get('OPENAI_API_KEY', '')
GROQ_KEY   = os.environ.get('GROQ_API_KEY', '')

if not OPENAI_KEY and not GROQ_KEY:
    print('Set OPENAI_API_KEY or GROQ_API_KEY in Cell 5 to enable the judge.')
else:
    !python evaluate_pipeline3.py \
        --backend local \
        --models biomistral kg kg_pure \
        --kg_path medical_kg_v3.json \
        --n_samples 50 \
        --no_bertscore \
        --llm_judge \
        --output_dir results_with_judge

In [ ]:
# ── Cell 12: Show results table ───────────────────────────────────────────────
import glob, os

tables = sorted(glob.glob('results/*.txt') + glob.glob('results_with_bertscore/*.txt'))
if tables:
    latest = tables[-1]
    print(f'Showing: {latest}\n')
    with open(latest) as f:
        print(f.read())
else:
    print('No results table found — run Cell 8 first.')

In [ ]:
# ── Cell 13: Download results ─────────────────────────────────────────────────
import glob
from google.colab import files

for f in glob.glob('results*/*.csv') + glob.glob('results*/*.json') + glob.glob('results*/*.txt'):
    print(f'Downloading: {f}')
    files.download(f)